# Preprocessing Experiments

Runs E1–E7 on three explicitly selected small videos. It measures processing time, cache behavior, compressed frame-cache storage, observed process RSS, resolution and errors. Results are pilot measurements and do not estimate full-dataset costs or QA accuracy.


In [ ]:
import json, subprocess, sys
from pathlib import Path
command=[sys.executable,"scripts/run_preprocessing_experiments.py","--limit-videos","3"]
completed=subprocess.run(command,text=True,capture_output=True)
print(completed.stdout)
if completed.returncode:
    print(completed.stderr)
    raise RuntimeError(f"Experiment runner failed with exit status {completed.returncode}")


In [ ]:
runs=sorted(Path("reports/preprocessing_runs").glob("*/results.json"),key=lambda p:p.stat().st_mtime)
result=json.loads(runs[-1].read_text(encoding="utf-8"))
experiment_df=pd.DataFrame([{**{k:x[k] for k in ("experiment","frames","strategy","successfully_processed_videos","storage_bytes","cache_hit_rate","peak_observed_process_rss_bytes")},
                             "first_pass_seconds":x["pass_timings"][0]["elapsed_seconds"],
                             "seconds_per_video":x["pass_timings"][0]["seconds_per_success"]}
                            for x in result["experiments"]])
experiment_df


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,4))
experiment_df.plot(x="experiment",y="seconds_per_video",kind="bar",legend=False,ax=axes[0],title="First-pass seconds per video")
experiment_df.plot(x="experiment",y="storage_bytes",kind="bar",legend=False,ax=axes[1],title="Sample cache storage (bytes)")
experiment_df.plot(x="experiment",y="peak_observed_process_rss_bytes",kind="bar",legend=False,ax=axes[2],title="Observed process RSS (bytes)")
plt.tight_layout()
print("sample paths:",result["sample_video_paths"])
print("scope:",result["scope"])
